# Feature Engineering

<div style="text-align: justify">

The following notebook is dedicated to feature engineering for the  <b> Tau Supersymmetry </b> search analysis. Feature engineering involves preparing input features to enhance their quality and relevance for the ML-based analysis. A general data processing workflow is structured as follows:

</div>

<img src="../assets/data_processing.png" alt="Workflow" style="width: 60%; display: block; margin: 0 auto;"/>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis configuration |
| Load | `io.load_samples` | Read per-sample parquet files |
| Group | `merger.group_samples` | Categorise into data / background / signal |
| Split | `merger.split_mc_data` | Combine MC (background + signal) and separate from data |
| Class | `merger.assign_class` | Assign integer class labels |
| Flatten | `merger.dict_to_array` | Concatenate all samples into one array |
| Extract | `features.extract_feature` | Save eventOrigin and tau_n before deletion |
| Drop | `features.drop_features` | Remove cleaning, truth, weight and auxiliary fields |
| Rectify | `rectangularizer.rectangularize` | Pad jagged features and convert to DataFrame |
| Align | — | Restrict data columns to match MC |
| Fill | `rectangularizer.fill_padding` | Replace NaN padding values |
| Reattach | — | Add eventOrigin and tau_n back to MC DataFrame |
| Weights | `features.assign_class_weights` | Compute and attach per-class sample weights to MC |
| Save | `io.save_dataframe` | Write mc.parquet and data.parquet |

The same pipeline is available as a CLI via `python run.py stage=feature_engineer` or `make feature-engineer`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data Processing:
* [Awkward Array](https://awkward-array.readthedocs.io/en/latest/)
* [Pandas](https://pandas.pydata.org/)

Serialization:
* [Apache Parquet](https://parquet.apache.org/)

### Notebook

Activating autoreload of imported modules.

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

Initializing the project root.

In [2]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [3]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration. All analysis parameters (run, region, channel, samples, features) are defined in `configs/` and can be overridden here.

In [4]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config")

Resolving the input directory from config.

In [5]:
from src.processing.analysis import get_output_paths

input_dir = path / get_output_paths(cfg)["samples_dir"]

## Deserialization

Loading all per-sample parquet files produced by the preprocessing pipeline.

In [6]:
from src.processing.io import load_samples

sample_ids = [f.stem for f in input_dir.glob("*.parquet")]
samples = load_samples(input_dir, sample_ids)

In [7]:
samples

{'data': <Array [{eventClean: 1, ...}, ..., {...}] type='319472 * {eventClean: int8,...'>,
 'topquarks': <Array [{eventClean: 1, ...}, ..., {...}] type='1139269 * {eventClean: int8...'>,
 'wtaunu': <Array [{eventClean: 1, ...}, ..., {...}] type='1716897 * {eventClean: int8...'>,
 'ztautau': <Array [{eventClean: 1, ...}, ..., {...}] type='2314192 * {eventClean: int8...'>,
 'diboson': <Array [{eventClean: 1, ...}, ..., {...}] type='1846720 * {eventClean: int8...'>,
 'signal': <Array [{eventClean: 1, ...}, ..., {...}] type='1986073 * {eventClean: int8...'>,
 'other': <Array [{eventClean: 1, ...}, ..., {...}] type='824018 * {eventClean: int8,...'>}

Grouping samples into data, background, and signal categories.

In [8]:
from src.processing.merger import group_samples

grouped = group_samples(samples, cfg)

In [9]:
grouped

{'data': {'data': <Array [{eventClean: 1, ...}, ..., {...}] type='319472 * {eventClean: int8,...'>},
 'background': {'topquarks': <Array [{eventClean: 1, ...}, ..., {...}] type='1139269 * {eventClean: int8...'>,
  'wtaunu': <Array [{eventClean: 1, ...}, ..., {...}] type='1716897 * {eventClean: int8...'>,
  'ztautau': <Array [{eventClean: 1, ...}, ..., {...}] type='2314192 * {eventClean: int8...'>,
  'diboson': <Array [{eventClean: 1, ...}, ..., {...}] type='1846720 * {eventClean: int8...'>,
  'other': <Array [{eventClean: 1, ...}, ..., {...}] type='824018 * {eventClean: int8,...'>},
 'signal': {'signal': <Array [{eventClean: 1, ...}, ..., {...}] type='1986073 * {eventClean: int8...'>}}

## Feature Engineering

Splitting into Monte-Carlo (background + signal) and data.

In [10]:
from src.processing.merger import split_mc_data

samples_mc, samples_data = split_mc_data(grouped)

In [11]:
samples_data

{'data': <Array [{eventClean: 1, ...}, ..., {...}] type='319472 * {eventClean: int8,...'>}

In [12]:
samples_mc

{'topquarks': <Array [{eventClean: 1, ...}, ..., {...}] type='1139269 * {eventClean: int8...'>,
 'wtaunu': <Array [{eventClean: 1, ...}, ..., {...}] type='1716897 * {eventClean: int8...'>,
 'ztautau': <Array [{eventClean: 1, ...}, ..., {...}] type='2314192 * {eventClean: int8...'>,
 'diboson': <Array [{eventClean: 1, ...}, ..., {...}] type='1846720 * {eventClean: int8...'>,
 'other': <Array [{eventClean: 1, ...}, ..., {...}] type='824018 * {eventClean: int8,...'>,
 'signal': <Array [{eventClean: 1, ...}, ..., {...}] type='1986073 * {eventClean: int8...'>}

### Class

Assigning an integer class label to each MC sample for ML training.

In [13]:
from src.processing.merger import assign_class                                        

assign_class(samples_mc)   

In [14]:
samples_mc['topquarks']['class']

<Array [0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0] type='1139269 * int64'>

Concatenating all samples into a single flat array.

In [15]:
from src.processing.merger import dict_to_array

samples_mc = dict_to_array(samples_mc)
samples_data = dict_to_array(samples_data)

In [16]:
samples_data

<Array [{eventClean: 1, ...}, ..., {...}] type='319472 * {eventClean: int8,...'>

In [17]:
samples_mc

<Array [{eventClean: 1, ...}, ..., {...}] type='9827169 * {eventClean: int8...'>

### Feature Extraction

Extracting features that will be needed after rectangularization.

In [18]:
from src.processing.features import extract_feature

event_origin = extract_feature(
    array_in=samples_mc,
    feature_name="eventOrigin",
)

tau_n = extract_feature(
    array_in=samples_mc,
    feature_name="tau_n",
)

In [19]:
event_origin

<Array ['topquarks', ..., 'SS_900_870_J85_1tau'] type='9827169 * string'>

In [20]:
tau_n

<Array [1, 1, 1, 1, 1, 1, 1, ..., 1, 1, 1, 1, 1, 1, 1] type='9827169 * int32'>

### Feature Deletion

Deleting cleaning, weight, truth, and auxiliary features.

In [21]:
from src.processing.features import drop_features, resolve_features_to_drop

features_to_drop = resolve_features_to_drop(cfg)

samples_mc = drop_features(array_in=samples_mc, feature_list=features_to_drop)
samples_data = drop_features(array_in=samples_data, feature_list=features_to_drop)

### Rectangularization & Padding

Kinematic data (mainly momenta and related quantities) is heavily nested / jagged. Most of the ML algorithms do not work well with such complicated arrays. Hence, the rectangular format is preferred. The current solution is to set up a certain threshold for a number of entries (jets) each event can take and pad them with zeros. An example is presented below:

<img src="../assets/padding_scheme.png" alt="Workflow" style="width: 60%; display: block; margin: 0 auto;"/>

Rectangularizing with parameters from `configs/data/default.yaml`:
* padding depth: `cfg.data.padding_threshold` (default 3 particles),
* MC: keeping features with at least `cfg.data.nan_threshold` (default 50%) non-missing values,
* data: keeping all features, then aligning to the MC column set.

In [22]:
from src.processing.rectangularizer import rectangularize

In [23]:
df_data = rectangularize(
    array_in=samples_data,
    padding_threshold=cfg.data.padding_threshold,
    nan_threshold=0.0,
)

In [24]:
df_mc = rectangularize(
    array_in=samples_mc,
    padding_threshold=cfg.data.padding_threshold,
    nan_threshold=cfg.data.nan_threshold,
)

Aligning data columns to MC.

In [25]:
df_data = df_data[df_mc.columns.intersection(df_data.columns)]

In [26]:
df_data

,nVtx,LeptonVeto,jet_n,jet_n_btag,sumMTJet,sumMTTauJet,met,met_phi,METSig,meff,...,jet_isBjet_2,jet_jvt_0,jet_jvt_1,jet_jvt_2,jet_delPhiMet_0,jet_delPhiMet_1,jet_delPhiMet_2,jet_width_0,jet_width_1,jet_width_2
entry,,,,,,,,,,,,,,,,,,,,,
0,9,True,3,0,7.108729e+05,7.344439e+05,204443.343750,-2.174925,10.849691,5.796566e+05,...,0.0,0.998978,0.287870,0.738909,3.060322,1.348413,1.924374,0.041627,0.066422,0.094496
1,11,True,2,1,5.434759e+05,5.737474e+05,230329.437500,1.430501,12.173761,4.510882e+05,...,NaN,0.993353,0.942097,NaN,2.598477,3.073735,NaN,0.029026,0.054394,NaN
2,7,True,3,0,8.190469e+05,9.522723e+05,234904.656250,-0.383476,14.782340,5.961410e+05,...,0.0,0.990696,0.987549,0.976860,2.426746,2.250051,1.732608,0.058812,0.046056,0.126268
3,11,True,4,2,1.439032e+06,1.481191e+06,366111.625000,2.497280,15.521962,9.661421e+05,...,0.0,0.994606,0.000000,0.987559,2.641968,2.385689,3.068347,0.150948,0.004699,0.139443
4,7,True,2,0,8.038616e+05,8.044959e+05,227796.421875,1.036523,11.347968,9.703381e+05,...,NaN,0.500902,1.000000,NaN,2.828401,0.666517,NaN,0.019953,0.055431,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319467,27,True,7,1,1.679186e+06,1.713091e+06,303796.531250,-0.397331,8.261763,1.537722e+06,...,0.0,0.982065,0.990000,0.996971,2.980759,0.420255,0.211701,0.186468,0.078968,0.119133
319468,20,True,2,0,6.648238e+05,6.970295e+05,242897.906250,1.215237,11.410558,5.362158e+05,...,NaN,0.970381,1.000000,NaN,3.074824,2.600345,NaN,0.034268,0.107167,NaN
319469,15,True,5,0,1.352135e+06,1.438104e+06,320696.218750,-0.561930,12.728176,9.029852e+05,...,0.0,1.000000,0.998038,0.948864,2.967621,2.997014,1.023377,0.080863,0.032645,0.160917


In [27]:
df_mc

,nVtx,LeptonVeto,jet_n,jet_n_btag,sumMTJet,sumMTTauJet,met,met_phi,METSig,meff,...,jet_isBjet_2,jet_jvt_0,jet_jvt_1,jet_jvt_2,jet_delPhiMet_0,jet_delPhiMet_1,jet_delPhiMet_2,jet_width_0,jet_width_1,jet_width_2
entry,,,,,,,,,,,,,,,,,,,,,
0,18,True,4,2,993838.375,1.035837e+06,2.512104e+05,-1.727673,12.867966,6.483069e+05,...,1.0,0.997757,0.997501,0.994129,2.684723,2.335376,2.842570,0.056821,0.076281,0.042212
1,11,True,4,2,1129536.375,1.160733e+06,2.425238e+05,1.655179,12.812569,7.935587e+05,...,0.0,-0.100000,0.999785,0.994528,2.592552,1.729147,3.038795,0.056552,0.063856,0.196646
2,17,True,7,2,1715605.625,1.730615e+06,2.086409e+05,-0.276374,8.978572,1.320639e+06,...,1.0,0.995554,0.986014,0.995531,2.131062,2.642069,0.417061,0.051609,0.075280,0.168994
3,10,True,5,2,1062983.750,1.126216e+06,2.068697e+05,2.280369,11.774617,6.442722e+05,...,1.0,0.992771,0.993609,0.990000,2.995725,1.983203,1.282939,0.081907,0.150282,0.050103
4,27,True,4,0,896357.375,9.441957e+05,2.095323e+05,-3.135049,11.435131,5.574934e+05,...,0.0,0.991493,0.991029,0.983820,3.061255,2.934011,3.036008,0.090903,0.092126,0.113758
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9827164,18,True,2,0,635148.375,7.503884e+05,2.780631e+05,-1.159799,13.717712,6.100386e+05,...,NaN,0.935357,0.812845,NaN,3.130677,0.911731,NaN,0.025179,0.128983,NaN
9827165,22,True,6,0,3874650.250,3.925604e+06,1.003722e+06,-0.666699,24.657297,2.145574e+06,...,0.0,1.000000,0.993866,0.997932,2.985127,3.046724,2.751268,0.089573,0.077406,0.175705
9827166,28,True,6,0,2673043.000,2.775120e+06,6.822745e+05,-1.799938,21.382816,1.981064e+06,...,0.0,0.988491,0.993427,0.989980,2.851266,0.840484,2.365701,0.022959,0.069122,0.140405


### Padding Fill

Replacing padding values with a fixed value or per-column mean.
* `NaN`,
* `0`,
* `-999`,
* `mean`.

Replacing NaN padding values with a 0.

In [28]:
from src.processing.rectangularizer import fill_padding

df_mc = fill_padding(df=df_mc, strategy="0")
df_data = fill_padding(df=df_data, strategy="0")

### Reattach Metadata

Attaching `eventOrigin` and `tau_n` back to the MC DataFrame — excluded from ML training but needed for downstream analysis and plotting.

In [29]:
df_mc['eventOrigin'] = event_origin
df_mc['tau_n'] = tau_n

### Class Weights

Computing per-class sample weights from the full MC dataset to correct for class imbalance during training.

In [30]:
from src.processing.features import assign_class_weights

assign_class_weights(df_mc)

class
0    0.723287
1    0.479946
2    0.356072
3    0.446206
4    1.000000
5    0.414898
Name: count, dtype: float64

### Preview

Previewing the processed rectangular data as a dataframe.

In [31]:
df_data

,nVtx,LeptonVeto,jet_n,jet_n_btag,sumMTJet,sumMTTauJet,met,met_phi,METSig,meff,...,jet_isBjet_2,jet_jvt_0,jet_jvt_1,jet_jvt_2,jet_delPhiMet_0,jet_delPhiMet_1,jet_delPhiMet_2,jet_width_0,jet_width_1,jet_width_2
entry,,,,,,,,,,,,,,,,,,,,,
0,9,True,3,0,7.108729e+05,7.344439e+05,204443.343750,-2.174925,10.849691,5.796566e+05,...,0.0,0.998978,0.287870,0.738909,3.060322,1.348413,1.924374,0.041627,0.066422,0.094496
1,11,True,2,1,5.434759e+05,5.737474e+05,230329.437500,1.430501,12.173761,4.510882e+05,...,0.0,0.993353,0.942097,0.000000,2.598477,3.073735,0.000000,0.029026,0.054394,0.000000
2,7,True,3,0,8.190469e+05,9.522723e+05,234904.656250,-0.383476,14.782340,5.961410e+05,...,0.0,0.990696,0.987549,0.976860,2.426746,2.250051,1.732608,0.058812,0.046056,0.126268
3,11,True,4,2,1.439032e+06,1.481191e+06,366111.625000,2.497280,15.521962,9.661421e+05,...,0.0,0.994606,0.000000,0.987559,2.641968,2.385689,3.068347,0.150948,0.004699,0.139443
4,7,True,2,0,8.038616e+05,8.044959e+05,227796.421875,1.036523,11.347968,9.703381e+05,...,0.0,0.500902,1.000000,0.000000,2.828401,0.666517,0.000000,0.019953,0.055431,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319467,27,True,7,1,1.679186e+06,1.713091e+06,303796.531250,-0.397331,8.261763,1.537722e+06,...,0.0,0.982065,0.990000,0.996971,2.980759,0.420255,0.211701,0.186468,0.078968,0.119133
319468,20,True,2,0,6.648238e+05,6.970295e+05,242897.906250,1.215237,11.410558,5.362158e+05,...,0.0,0.970381,1.000000,0.000000,3.074824,2.600345,0.000000,0.034268,0.107167,0.000000
319469,15,True,5,0,1.352135e+06,1.438104e+06,320696.218750,-0.561930,12.728176,9.029852e+05,...,0.0,1.000000,0.998038,0.948864,2.967621,2.997014,1.023377,0.080863,0.032645,0.160917


In [32]:
df_mc

,nVtx,LeptonVeto,jet_n,jet_n_btag,sumMTJet,sumMTTauJet,met,met_phi,METSig,meff,...,jet_jvt_2,jet_delPhiMet_0,jet_delPhiMet_1,jet_delPhiMet_2,jet_width_0,jet_width_1,jet_width_2,eventOrigin,tau_n,class_weight
entry,,,,,,,,,,,,,,,,,,,,,
0,18,True,4,2,993838.375,1.035837e+06,2.512104e+05,-1.727673,12.867966,6.483069e+05,...,0.994129,2.684723,2.335376,2.842570,0.056821,0.076281,0.042212,topquarks,1,0.723287
1,11,True,4,2,1129536.375,1.160733e+06,2.425238e+05,1.655179,12.812569,7.935587e+05,...,0.994528,2.592552,1.729147,3.038795,0.056552,0.063856,0.196646,topquarks,1,0.723287
2,17,True,7,2,1715605.625,1.730615e+06,2.086409e+05,-0.276374,8.978572,1.320639e+06,...,0.995531,2.131062,2.642069,0.417061,0.051609,0.075280,0.168994,topquarks,1,0.723287
3,10,True,5,2,1062983.750,1.126216e+06,2.068697e+05,2.280369,11.774617,6.442722e+05,...,0.990000,2.995725,1.983203,1.282939,0.081907,0.150282,0.050103,topquarks,1,0.723287
4,27,True,4,0,896357.375,9.441957e+05,2.095323e+05,-3.135049,11.435131,5.574934e+05,...,0.983820,3.061255,2.934011,3.036008,0.090903,0.092126,0.113758,topquarks,1,0.723287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9827164,18,True,2,0,635148.375,7.503884e+05,2.780631e+05,-1.159799,13.717712,6.100386e+05,...,0.000000,3.130677,0.911731,0.000000,0.025179,0.128983,0.000000,SS_900_870_J85_1tau,1,0.414898
9827165,22,True,6,0,3874650.250,3.925604e+06,1.003722e+06,-0.666699,24.657297,2.145574e+06,...,0.997932,2.985127,3.046724,2.751268,0.089573,0.077406,0.175705,SS_900_870_J85_1tau,1,0.414898
9827166,28,True,6,0,2673043.000,2.775120e+06,6.822745e+05,-1.799938,21.382816,1.981064e+06,...,0.989980,2.851266,0.840484,2.365701,0.022959,0.069122,0.140405,SS_900_870_J85_1tau,1,0.414898


## Serialization

Saving MC and data dataframes as parquet files to the dataframes directory.

In [33]:
from src.processing.analysis import get_output_paths
from src.processing.io import save_dataframe

dataframes_dir = path / get_output_paths(cfg)["dataframes_dir"]

In [34]:
save_dataframe(df=df_mc, path=dataframes_dir / "mc.parquet")
save_dataframe(df=df_data, path=dataframes_dir / "data.parquet")